In [1]:
!pip install ultralytics opencv-python scipy numpy

import cv2
import numpy as np
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [10]:
class Config:
    CONF_THRESHOLD = 0.4
    NMS_THRESHOLD = 0.5
    IOU_THRESHOLD = 0.3
    MAX_AGE = 30
    MIN_HITS = 3
    PROCESS_NOISE = 1e-2
    MEASUREMENT_NOISE = 1e-1

In [11]:
class Detector:
    def __init__(self, model_path="yolov8n.pt"):
        self.model = YOLO(model_path)

    def detect(self, frame):
        results = self.model(frame, conf=Config.CONF_THRESHOLD, iou=Config.NMS_THRESHOLD)[0]

        detections = []
        for box in results.boxes:
            cls = int(box.cls[0])

            if cls != 0:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            w = x2 - x1
            h = y2 - y1

            detections.append([x1, y1, w, h])

        return np.array(detections)

In [12]:
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)

    inter_area = inter_w * inter_h
    union_area = boxA[2]*boxA[3] + boxB[2]*boxB[3] - inter_area

    if union_area == 0:
        return 0

    return inter_area / union_area

In [13]:
class KalmanFilter:
    def __init__(self, bbox):
        x, y, w, h = bbox

        self.x = np.array([x, y, w, h, 0, 0], dtype=float)

        self.P = np.eye(6)
        self.F = np.eye(6)
        self.F[0, 4] = 1
        self.F[1, 5] = 1

        self.H = np.zeros((4, 6))
        self.H[0, 0] = 1
        self.H[1, 1] = 1
        self.H[2, 2] = 1
        self.H[3, 3] = 1

        self.Q = Config.PROCESS_NOISE * np.eye(6)
        self.R = Config.MEASUREMENT_NOISE * np.eye(4)

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x[:4]

    def update(self, z):
        z = np.array(z)

        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        self.x = self.x + K @ y
        self.P = (np.eye(6) - K @ self.H) @ self.P

    def get_state(self):
        return self.x[:4]

In [14]:
class Track:
    count = 0

    def __init__(self, bbox):
        self.id = Track.count
        Track.count += 1
        self.kf = KalmanFilter(bbox)

        self.age = 1
        self.hits = 1
        self.time_since_update = 0
        self.confirmed = False

    def predict(self):
        self.kf.predict()
        self.age += 1
        self.time_since_update += 1

    def update(self, bbox):
        self.kf.update(bbox)
        self.hits += 1
        self.time_since_update = 0

        if self.hits >= Config.MIN_HITS:
            self.confirmed = True

    def get_bbox(self):
        return self.kf.get_state()

In [15]:
def associate_detections_to_tracks(detections, tracks):
    if len(tracks) == 0:
        return [], list(range(len(detections))), []

    cost_matrix = np.zeros((len(tracks), len(detections)))

    for t, track in enumerate(tracks):
        for d, det in enumerate(detections):
            iou = compute_iou(track.get_bbox(), det)
            cost_matrix[t, d] = 1 - iou

    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    matches, unmatched_tracks, unmatched_dets = [], [], []

    for t in range(len(tracks)):
        if t not in row_ind:
            unmatched_tracks.append(t)

    for d in range(len(detections)):
        if d not in col_ind:
            unmatched_dets.append(d)

    for r, c in zip(row_ind, col_ind):
        if 1 - cost_matrix[r, c] < Config.IOU_THRESHOLD:
            unmatched_tracks.append(r)
            unmatched_dets.append(c)
        else:
            matches.append((r, c))

    return matches, unmatched_dets, unmatched_tracks

In [16]:
class Tracker:
    def __init__(self):
        self.tracks = []

    def update(self, detections):

        for track in self.tracks:
            track.predict()

        matches, unmatched_dets, unmatched_tracks = associate_detections_to_tracks(
            detections, self.tracks
        )

        for t, d in matches:
            self.tracks[t].update(detections[d])

        for i in unmatched_dets:
            self.tracks.append(Track(detections[i]))

        self.tracks = [
            t for t in self.tracks if t.time_since_update < Config.MAX_AGE
        ]

        return self.tracks

In [17]:
def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)

    width = int(cap.get(3))
    height = int(cap.get(4))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"XVID"),
        fps,
        (width, height)
    )

    detector = Detector()
    tracker = Tracker()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        detections = detector.detect(frame)
        tracks = tracker.update(detections)

        for track in tracks:
            if not track.confirmed:
                continue
            x, y, w, h = map(int, track.get_bbox())

            cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
            cv2.putText(frame, f"ID {track.id}",
                        (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0,255,0), 2)

        out.write(frame)
    cap.release()
    out.release()

In [18]:
process_video("/content/tracking_train.avi", "/content/train_output.avi")


0: 448x640 6 persons, 78.1ms
Speed: 12.5ms preprocess, 78.1ms inference, 39.9ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 7 persons, 6.6ms
Speed: 3.2ms preprocess, 6.6ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 7 persons, 6.4ms
Speed: 2.3ms preprocess, 6.4ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 7 persons, 6.3ms
Speed: 2.3ms preprocess, 6.3ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 6 persons, 8.5ms
Speed: 2.7ms preprocess, 8.5ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 7 persons, 6.1ms
Speed: 2.2ms preprocess, 6.1ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 7 persons, 7.8ms
Speed: 1.6ms preprocess, 7.8ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 6 persons, 8.1ms
Speed: 2.7ms preprocess, 8.1ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 6

In [19]:
process_video("/content/tracking_test.avi", "/content/test_output.avi")


0: 448x640 9 persons, 6.5ms
Speed: 1.7ms preprocess, 6.5ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 8 persons, 6.0ms
Speed: 2.3ms preprocess, 6.0ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 8 persons, 6.3ms
Speed: 2.2ms preprocess, 6.3ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 8 persons, 6.3ms
Speed: 2.3ms preprocess, 6.3ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 10 persons, 6.8ms
Speed: 1.9ms preprocess, 6.8ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 9 persons, 6.0ms
Speed: 2.0ms preprocess, 6.0ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 9 persons, 6.0ms
Speed: 2.2ms preprocess, 6.0ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 9 persons, 6.0ms
Speed: 2.3ms preprocess, 6.0ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)

In [20]:
from IPython.display import Video

Video("/content/train_output.avi")

In [22]:
Video("/content/test_output.avi")